In [ ]:
!pip install pennylane torch torchvision scikit-learn pandas
!pip install rdkit

In [ ]:
#We predict the toxicity of chemical compounds across multiple biological targets using multi-label classification.

In [ ]:
\



!wget https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz

import pandas as pd
df = pd.read_csv('tox21.csv.gz')
print(df.head())

--2026-04-02 08:53:46--  https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz
Resolving deepchemdata.s3-us-west-1.amazonaws.com (deepchemdata.s3-us-west-1.amazonaws.com)... 52.219.192.66, 16.15.2.204, 52.219.220.18, ...
Connecting to deepchemdata.s3-us-west-1.amazonaws.com (deepchemdata.s3-us-west-1.amazonaws.com)|52.219.192.66|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 122925 (120K) [application/x-gzip]
Saving to: ‘tox21.csv.gz.3’

tox21.csv.gz.3      100%[===================>] 120.04K  --.-KB/s    in 0.05s   

2026-04-02 08:53:46 (2.35 MB/s) - ‘tox21.csv.gz.3’ saved [122925/122925]

   NR-AR  NR-AR-LBD  NR-AhR  NR-Aromatase  NR-ER  NR-ER-LBD  NR-PPAR-gamma  \
0    0.0        0.0     1.0           NaN    NaN        0.0            0.0   
1    0.0        0.0     0.0           0.0    0.0        0.0            0.0   
2    NaN        NaN     NaN           NaN    NaN        NaN            NaN   
3    0.0        0.0     0.0           0.0    0.0

In [ ]:
print(df.columns)
print(df.shape)

Index(['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
       'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
       'mol_id', 'smiles'],
      dtype='object')
(7831, 14)


In [ ]:
df = df[['smiles', 'NR-AR']].dropna()
df.columns = ['smiles', 'label']

print(df.head())
print(df.shape)

                                 smiles  label
0          CCOc1ccc2nc(S(N)(=O)=O)sc2c1    0.0
1             CCN1C(=O)NC(c2ccccc2)C1=O    0.0
3       CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C    0.0
4             CC(O)(P(=O)(O)O)P(=O)(O)O    0.0
5  CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C    0.0
(7265, 2)


In [ ]:
#feature extraction
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np

def featurize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    return [
        Descriptors.MolWt(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.MolLogP(mol)
    ]

df['features'] = df['smiles'].apply(featurize)
df = df.dropna()

X = np.array(df['features'].tolist())
y = df['label'].values

print("Feature shape:", X.shape)
print("Label shape:", y.shape)

[08:53:47] WARNING: not removing hydrogen atom without neighbors
[08:53:49] Explicit valence for atom # 3 Al, 6, is greater than permitted
[08:53:49] Explicit valence for atom # 4 Al, 6, is greater than permitted
[08:53:52] Explicit valence for atom # 4 Al, 6, is greater than permitted
[08:53:53] Explicit valence for atom # 9 Al, 6, is greater than permitted
[08:53:53] Explicit valence for atom # 5 Al, 6, is greater than permitted
[08:53:54] Explicit valence for atom # 16 Al, 6, is greater than permitted
[08:53:56] Explicit valence for atom # 20 Al, 6, is greater than permitted


Feature shape: (7258, 4)
Label shape: (7258,)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reduce dimensions (IMPORTANT for quantum)
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5806, 4)
Test shape: (1452, 4)


In [ ]:
#Build Quantum Circuit (Core Innovation) We will create a Variational Quantum Circuit (QNN).
#What this does Converts classical data → quantum states Learns patterns using: Superposition Entanglement

import pennylane as qml
from pennylane import numpy as np

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    for i in range(n_qubits):
        qml.RZ(weights[i], wires=i)

    for i in range(n_qubits - 1):
        qml.CNOT(wires=[i, i+1])

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [ ]:
import torch
import torch.nn as nn

class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(4, 4)

        weight_shapes = {"weights": (4,)}
        self.q_layer = qml.qnn.TorchLayer(quantum_circuit, weight_shapes)

        self.fc2 = nn.Linear(4, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))

        # 🔥 FIX: process each sample separately
        q_out = []
        for i in range(x.shape[0]):
            q_out.append(self.q_layer(x[i]))

        x = torch.stack(q_out)

        x = self.fc2(x)
        return torch.sigmoid(x)

In [ ]:
from sklearn.utils import resample

# Combine X and y
import numpy as np
data = np.hstack((X_train, y_train.reshape(-1,1)))

# Separate classes
majority = data[data[:,-1]==0]
minority = data[data[:,-1]==1]

# Upsample minority
minority_upsampled = resample(minority,
                             replace=True,
                             n_samples=len(majority),
                             random_state=42)

# Combine
balanced = np.vstack((majority, minority_upsampled))

X_train_bal = balanced[:,:-1]
y_train_bal = balanced[:,-1]

# Convert to tensor
X_train_t = torch.tensor(X_train_bal[:1000], dtype=torch.float32)
y_train_t = torch.tensor(y_train_bal[:1000], dtype=torch.float32).view(-1,1)

In [22]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.fc2 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return self.sigmoid(x)

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Initialize model
model = SimpleModel()

optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCELoss()

# Convert data to tensors
#  (use balanced data)
X_train_t = torch.tensor(X_train_bal, dtype=torch.float32)
y_train_t = torch.tensor(y_train_bal, dtype=torch.float32).view(-1, 1)

# Create DataLoader (batching)
dataset = TensorDataset(X_train_t, y_train_t)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Training loop
for epoch in range(20):
    total_loss = 0
    total_acc = 0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = loss_fn(outputs, y_batch)

        loss.backward()
        optimizer.step()

        preds = (outputs > 0.5).float()
        acc = (preds == y_batch).float().mean()

        total_loss += loss.item()
        total_acc += acc.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Acc: {total_acc/len(loader):.4f}")


Epoch 1, Loss: 213.3272, Acc: 0.6964
Epoch 2, Loss: 204.3299, Acc: 0.7181
Epoch 3, Loss: 201.6995, Acc: 0.7253
Epoch 4, Loss: 200.0999, Acc: 0.7253
Epoch 5, Loss: 198.7839, Acc: 0.7274
Epoch 6, Loss: 197.5921, Acc: 0.7273
Epoch 7, Loss: 197.4614, Acc: 0.7256
Epoch 8, Loss: 197.3677, Acc: 0.7284
Epoch 9, Loss: 197.3748, Acc: 0.7284
Epoch 10, Loss: 196.8650, Acc: 0.7281
Epoch 11, Loss: 196.7639, Acc: 0.7272
Epoch 12, Loss: 196.6599, Acc: 0.7300
Epoch 13, Loss: 196.7455, Acc: 0.7287
Epoch 14, Loss: 196.8332, Acc: 0.7302
Epoch 15, Loss: 196.4937, Acc: 0.7322
Epoch 16, Loss: 196.5073, Acc: 0.7328
Epoch 17, Loss: 196.1928, Acc: 0.7308
Epoch 18, Loss: 196.5059, Acc: 0.7303
Epoch 19, Loss: 197.1544, Acc: 0.7304
Epoch 20, Loss: 196.7347, Acc: 0.7305


In [24]:
 # ✅ SAVE MODEL HERE (after training ends)
torch.save(model.state_dict(), "simple_model.pth")
print("✅ Model saved successfully!")

✅ Model saved successfully!


In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score

X_test_t = torch.tensor(X_test, dtype=torch.float32)

# Predictions
probs = model(X_test_t).detach().numpy()
preds = (probs > 0.7).astype(int)

# Metrics
accuracy = accuracy_score(y_test, preds)
precision = precision_score(y_test, preds)
recall = recall_score(y_test, preds)
f1 = f1_score(y_test, preds)
roc_auc = roc_auc_score(y_test, probs)

# Confusion Matrix
cm = confusion_matrix(y_test, preds)

print("🔹 Accuracy :", accuracy)

print("🔹 ROC-AUC  :", roc_auc)

print("\nConfusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_test, preds))

🔹 Accuracy : 0.8546831955922864
🔹 ROC-AUC  : 0.6775672840769545

Confusion Matrix:
 [[1224  172]
 [  39   17]]

Classification Report:

              precision    recall  f1-score   support

         0.0       0.97      0.88      0.92      1396
         1.0       0.09      0.30      0.14        56

    accuracy                           0.85      1452
   macro avg       0.53      0.59      0.53      1452
weighted avg       0.94      0.85      0.89      1452



In [26]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, probs)

# Find best F1
f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
best_thresh = thresholds[f1.argmax()]

print("Best Threshold:", best_thresh)

Best Threshold: 0.60741377


In [27]:
import os
print(os.listdir())

['.config', 'tox21.csv.gz.2', 'simple_model.pth', 'tox21.csv.gz.3', 'tox21.csv.gz', 'tox21.csv.gz.1', 'toxicity_model.pth', 'sample_data']


In [29]:
from google.colab import files
files.download( "simple_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>